In [ ]:
# low level
from enum import Enum
import os
import random
from time import time
import datetime
import math
 
# middle level
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import pandas as pd
import sklearn
 
# frameworkscompute the correlation coefficient between multiple variables python
import torch 
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms

# Data

In [ ]:
data = pd.read_csv("/content/framingham Dataset.csv")
data = data.dropna()

In [ ]:
data

## Normalizations


In [ ]:
to_dummy_data = data.copy()

In [ ]:
for d in ["age", "education", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]:
    to_dummy_data.pop(d)

In [ ]:
to_dummy_data

In [ ]:
to_normalize_data = data.copy()

In [ ]:
for d in ["male", "currentSmoker", "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes", "TenYearCHD"]:
    to_normalize_data.pop(d)

In [ ]:
to_normalize_data

In [ ]:
to_normalize_data_stats = to_normalize_data.describe()
to_normalize_data_stats = to_normalize_data_stats.transpose()
to_normalize_data_stats

In [ ]:
def norm(x):
  return (x - to_normalize_data_stats['mean']) / to_normalize_data_stats['std']
normalize_data = norm(to_normalize_data)

In [ ]:
normalize_data

In [ ]:
dummy_data = {}
for d in to_dummy_data:
    dummy_data[d] = pd.get_dummies(to_dummy_data[d])

In [ ]:
for d in dummy_data:
    print(d,'\n',dummy_data[d][:10])

## Reset Index

In [ ]:
normalize_data = normalize_data.reset_index()
for d in dummy_data:
    dummy_data[d] = dummy_data[d].reset_index()

## Data Class

In [ ]:
class DataClass(Dataset):
    def __init__(self, normalize_data, dummy_data, targed_features):
        self.normalize_data = normalize_data
        self.dummy_data = dummy_data
        self.targed_features = targed_features        
    def __len__(self):
        return len(self.normalize_data)
    
    def __getitem__(self, indx):
        inputs = []
        inputs_labels = []
        for d in self.normalize_data:
            if d not in self.targed_features:
                continue
            inputs.append(self.normalize_data[d][indx])
            inputs_labels.append(d)
        
        for d in self.dummy_data:
            if d not in self.targed_features:
                continue
            inputs.append(self.dummy_data[d][0][indx])
            inputs.append(self.dummy_data[d][1][indx])
            inputs_labels.append("not"+d)
            inputs_labels.append(d)
        outputs = [self.dummy_data["TenYearCHD"][0][indx], self.dummy_data["TenYearCHD"][1][indx]]
        return [inputs, outputs, inputs_labels]

In [ ]:
targeted_features =['age','education','cigsPerDay','totChol','sysBP','diaBP','BMI','heartRate','glucose','male','currentSmoker','BPMeds','prevalentStroke','prevalentHyp','diabetes',]

In [ ]:
d = DataClass(normalize_data,dummy_data, targeted_features)

In [ ]:
d.__getitem__(14)

In [ ]:
len(d) == len(dummy_data['male']) == len(normalize_data)

## Feature Importance

In [ ]:
data = DataClass(normalize_data,dummy_data,targeted_features)
inputs = []
outputs = []
for i in range(len(data)):
    input, output, input_labels = data.__getitem__(i)
    inputs.append(input)
    outputs.append(output)
    print(f"\r{int(((i+1)/len(data)) *100)} % {i}",end='')

In [ ]:
inputs = np.asarray(inputs)
outputs = np.asarray(outputs)

### LinearRegression

In [ ]:
# linear regression feature importance
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from matplotlib import pyplot

# define dataset
X, y = np.copy(inputs), np.copy(outputs)
print("done")
# define the model
model = LinearRegression()
# fit the model
model.fit(X, y)
# get importance
importance = model.coef_
# summarize feature importance
importance_list = []
for i,v in enumerate(importance[0]):
	importance_list.append([i,v])
	print('Feature: %0d, Score: %.5f' % (i,v))
# plot feature importance
pyplot.bar([x for x in range(len(importance[0]))], importance[0])

pyplot.show()

In [ ]:
y.shape

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from keras.models import Sequential
from keras.layers import Dense,Dropout
from keras.wrappers.scikit_learn import KerasClassifier

In [ ]:
def create_model():
  model = Sequential()
  model.add(Dense(12,activation='relu',input_dim = 21))
  model.add(Dense(2,activation='sigmoid'))
  model.compile(loss='binary_crossentropy',optimizer = 'adam',metrics=['accuracy'])
  return model

In [ ]:
# Fix random seed for reproducibillity
seed = 20
np.random.rand(seed)

In [ ]:
# Create model 
model = KerasClassifier(build_fn=create_model,verbose=0)

In [ ]:
#define GridSearch
batch_size = [10,20,40,60,80,100]
epochs = [10,50,100]

In [ ]:
param_grid = dict(batch_size=batch_size,epochs=epochs)

In [ ]:
grid = GridSearchCV(estimator=model,param_grid=param_grid,n_jobs=1,cv=3,verbose=0)

In [ ]:
grid_result = grid.fit(X,y)
#Summarize Result
print('Best : %f using : %s' %(grid_result.best_score_,grid_result.best_params_) )
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for means,stdev,params in zip(means,stds,params):
  print('%f (%f) with : %r'%(means,stdev,params))

In [ ]:
# Define Optimizer
def create_model(optimizer='adam'):
  model = Sequential()
  model.add(Dense(12,activation='relu',input_dim = 21))
  model.add(Dense(2,activation='sigmoid'))
  model.compile(loss='binary_crossentropy',optimizer = optimizer,metrics=['accuracy'])
  return model

In [ ]:
# Create model 
model = KerasClassifier(build_fn=create_model,epochs=10,batch_size=60,verbose=0)

In [ ]:
#define optimizer 
optimizer = ['SGD','Adam','Adamax','Nadam','RMSprop','Adadelta','Adagrad']

In [ ]:
param_grid_op = dict(optimizer=optimizer)

In [ ]:
grid = GridSearchCV(estimator=model,param_grid=param_grid_op,n_jobs=1,cv=3)

In [ ]:
grid_result = grid.fit(X,y)
#Summarize Result
print('Best : %f using : %s' %(grid_result.best_score_,grid_result.best_params_) )
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for means,stdev,params in zip(means,stds,params):
  print('%f (%f) with : %r'%(means,stdev,params))

In [ ]:
#define activation
def create_model(activation = 'relu'):
  model = Sequential()
  model.add(Dense(12,activation=activation,input_dim = 21))
  model.add(Dense(2,activation='sigmoid'))
  model.compile(loss='binary_crossentropy',optimizer = 'RMSprop',metrics=['accuracy'])
  return model

In [ ]:
# Create model 
model = KerasClassifier(build_fn=create_model,epochs=10,batch_size=60,verbose=0)

In [ ]:
# define gridsearch parameters
activation = ['relu','tanh','sigmoid','hard_sigmoid','softmax','softsign','linear','softplus']

In [ ]:
param_grid_ac = dict(activation = activation)

In [ ]:
grid = GridSearchCV(estimator=model,param_grid=param_grid_ac,n_jobs=1,cv=3,verbose=0)

In [ ]:
grid_result = grid.fit(X,y)
#Summarize Result
print('Best : %f using : %s' %(grid_result.best_score_,grid_result.best_params_) )
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for means,stdev,params in zip(means,stds,params):
  print('%f (%f) with : %r'%(means,stdev,params))

In [ ]:
# define dropout function
def create_model(dropout_rate = 0.0):
  model = Sequential()
  model.add(Dense(12,activation='linear',input_dim = 21))
  model.add(Dropout(dropout_rate))
  model.add(Dense(2,activation='sigmoid'))
  model.compile(loss='binary_crossentropy',optimizer = 'RMSprop',metrics=['accuracy'])
  return model

In [ ]:
model = KerasClassifier(build_fn=create_model,epochs = 10, batch_size=60,verbose=0)

In [ ]:
dropout_rate = [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

In [ ]:
param_grid_dr = dict(dropout_rate = dropout_rate)

In [ ]:
grid = GridSearchCV(estimator=model,param_grid=param_grid_dr,n_jobs=1,cv=3,verbose=0)

In [ ]:
grid_result = grid.fit(X,y)
#Summarize Result
print('Best : %f using : %s' %(grid_result.best_score_,grid_result.best_params_) )
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for means,stdev,params in zip(means,stds,params):
  print('%f (%f) with : %r'%(means,stdev,params))


In [ ]:
# define num of neurons 
def create_model(neurons = 1):
  model = Sequential()
  model.add(Dense(neurons,activation = 'linear',input_dim=21))
  model.add(Dropout(rate= 0.0))
  model.add(Dense(2,activation ='sigmoid'))
  model.compile(loss='binary_crossentropy',optimizer ='RMSprop',metrics=['accuracy'])
  return model

In [ ]:
model = KerasClassifier(build_fn=create_model,epochs = 10, batch_size=10,verbose=0)

In [ ]:
# define grid param
neurons = [1,10,15,20,28,32,64,128]

In [ ]:
param_grid_nu = dict(neurons = neurons)

In [ ]:
grid = GridSearchCV(estimator=model,param_grid=param_grid_nu,n_jobs=1,cv=3,verbose=0)

In [ ]:
grid_result = grid.fit(X,y)
#Summarize Result
print('Best : %f using : %s' %(grid_result.best_score_,grid_result.best_params_) )
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for means,stdev,params in zip(means,stds,params):
  print('%f (%f) with : %r'%(means,stdev,params))

In [ ]:
from keras.optimizers import SGD

In [ ]:
# define SGD Lr & momentum
def create_model(learn_rate = 0.01,momentum = 0.0):
  model = Sequential()
  model.add(Dense(10,activation = 'linear',input_dim=21))
  model.add(Dropout(rate=0.2))
  model.add(Dense(2,activation ='sigmoid'))
  opt = SGD(learning_rate=learn_rate,momentum=momentum)
  model.compile(loss='binary_crossentropy',optimizer =opt,metrics=['accuracy'])
  return model

In [ ]:
model = KerasClassifier(build_fn=create_model,epochs = 10, batch_size=60,verbose=0)

In [ ]:
learn_rate = [0.001,0.01,0.1,0.2,0.3]
momentum = [0.0,0.2,0.4,0.6,0.8,0.9]
param_grid_SGD = dict(learn_rate= learn_rate,momentum=momentum)

In [ ]:
grid = GridSearchCV(estimator=model,param_grid=param_grid_SGD,n_jobs=1,cv=3,verbose=0)

In [ ]:
grid_result = grid.fit(X,y)
#Summarize Result
print('Best : %f using : %s' %(grid_result.best_score_,grid_result.best_params_) )
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for means,stdev,params in zip(means,stds,params):
  print('%f (%f) with : %r'%(means,stdev,params))

In [ ]:
print('Best : 0.851559 using : batch_size: 60, epochs: 10')
print('Best : 0.850740 using : optimizer: RMSprop')
print('Best : 0.852926 using : activation: linear')
print('Best : 0.852927 using : dropout_rate: 0.0')
print('Best : 0.853746 using : neurons: 10')
print('Best : 0.854566 using : learn_rate: 0.2, momentum: 0.4')

In [ ]:
importance_list.sort(key=lambda x: abs(x[1]))

In [ ]:
x = []
# i = 0
for indx, score in importance_list[::-1]:
    if input_labels[indx][:3] != "not": 
        x.append(score)
    # i+=1
    # if i==11:
        # break
        print(indx, input_labels[indx], score)

In [ ]:
pyplot.bar([x for x in range(len(x))], x)

pyplot.show()

## Data Creator

In [ ]:
def create_datasets(normalize_data=normalize_data, dummy_data=dummy_data, targeted_features=targeted_features, batch_size=32, valid_size = 0.25):

    train_dataset = DataClass(normalize_data,dummy_data, targeted_features)
    valid_dataset = DataClass(normalize_data,dummy_data, targeted_features)

    train_size = len(train_dataset)

    indices = list(range(train_size))
    np.random.shuffle(indices)

    valid_split_size = int(valid_size * train_size)

    train_indices, valid_indices = indices[valid_split_size:], indices[:valid_split_size]

    train_sampler = SubsetRandomSampler(train_indices)
    valid_sampler = SubsetRandomSampler(valid_indices)
   
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=train_sampler)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size,sampler=valid_sampler)
    
    print(f"\nTrain set: {len(train_indices)} sample")
    print(f"Valid set: {len(valid_indices)} sample")


    return train_loader, valid_loader

In [ ]:
train_loader, valid_loader = create_datasets()

# Model

In [ ]:
class Model(nn.Module):
    def __init__(self, input_size=20, n=15):
        super(Model, self).__init__()
        self.fc1 = nn.Linear(input_size, n)
        self.fc2 = nn.Linear(n, n)
        self.fc3 = nn.Linear(n, n)
        self.fc4 = nn.Linear(n, 2)
        self.ReLU = nn.ReLU()
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.ReLU(x)
        x = self.fc2(x)
        x = self.ReLU(x)
        x = self.fc3(x)
        x = self.ReLU(x)
        x = self.fc4(x)
        return x

In [ ]:
model = Model(21)
model(torch.rand(1,21))

# Workers

## Early Stopper

In [ ]:
class EarlyStopping:
    def __init__(self,patience=15, path='/content'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
    
    def __call__(self, val_loss, model, epoch):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
        if score < self.best_score:
            self.counter += 1
            print(f'\rEpoch {epoch} EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            print("\rEpoch {} loss reduced from {} to {}".format(epoch, -self.best_score, val_loss))
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
    
    def save_checkpoint(self, val_loss, model):
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

## Training

In [ ]:
x = None
def train(train_loader, model, optimizer, criterion):
    global x
    model.train()
    t = time()
    train_loss = 0
    for i, data in enumerate(train_loader):
        inputs, outputs, labels = data

        l=[]
        for inp in inputs:
            l.append(inp.double())
        inputs = torch.stack(l,1)
        
        l=[]
        for out in outputs: 
            l.append(out.double())
        outputs = torch.stack(l,1)
        inputs = inputs.float()
        outputs = outputs.float()
        
        optimizer.zero_grad()
        
       
        _output = model(inputs)
        loss = criterion(_output, outputs)
        loss.backward()
        optimizer.step()
        
        train_loss += 1/(i + 1) * (loss.data - train_loss)
        

        torch.cuda.empty_cache()
        
        delay = time() - t
        hms = str(datetime.timedelta(seconds=int(delay* (len(train_loader)-(i+1)))))

        print(f"\rTaining finished batches {i+1}/{len(train_loader)}   {int(((i+1)/len(train_loader))*100)}%   delay {int(delay)}s   time left {hms}   loss {loss.item()}",end='')
        
        t = time()
        
    return train_loss

## Valid

In [ ]:
def validation(valid_loader, model, criterion):
    model.eval()
    loss, running_loss = 0.0, 0.0
    t = time()
    with torch.no_grad():
        for i, data in enumerate(valid_loader):
            inputs, outputs, labels = data

            l=[]
            for inp in inputs:
                l.append(inp.double())
            inputs = torch.stack(l,1)
            
            l=[]
            for out in outputs:
                l.append(out.double())
            outputs = torch.stack(l,1)
            
            inputs = inputs.float()
            outputs = outputs.float()

            _output = model(inputs)
            loss = criterion(_output, outputs)
            running_loss += loss.item()
         

            delay = time() - t
            hms = str(datetime.timedelta(seconds=int(delay * (len(valid_loader)-(i+1)))))
            print(f"\rValidation finished batches {i+1}/{len(valid_loader)}   {int(((i+1)/len(valid_loader))*100)}%   delay {int(delay)}s   time left {hms}   loss { loss.item()} ",end='')
            t = time()
    avg_loss = running_loss / (i+1)
    model.train()
    return avg_loss

## Worker

In [ ]:
def worker(n_epochs, train_loader, valid_loader, model, optimizer, criterion, early_stop):
    
    all_train_loss, all_val_loss  = [], []
    for epoch in range(n_epochs):
        train_loss = train(train_loader, model, optimizer, criterion)
        val_loss = validation(valid_loader, model, criterion)
        early_stop(val_loss, model, epoch)
        all_train_loss.append(train_loss)
        all_val_loss.append(val_loss)

        if early_stop.early_stop:
            break    
            
    return all_train_loss, all_val_loss  

# Start

In [ ]:
# optimizer = optim.SGD(model.parameters(), 0.01)
# criterion = nn.MSELoss()
# train(train_loader,model,optimizer,criterion);

In [ ]:
optimizer = optim.SGD(model.parameters(), 0.01)
criterion = nn.MSELoss()
validation(valid_loader,model,criterion);

In [ ]:
es = EarlyStopping(10,"model.pt")
all_train_loss, all_val_loss = worker(10000, train_loader, valid_loader, model, optimizer, criterion,es)

In [ ]:
def display_graph(train_losses, valid_losses):
    plt.plot(train_losses, label='Model pridictions')
    plt.plot(valid_losses, label='Real Valuse loss')

    plt.legend(frameon=False)

In [ ]:
display_graph(all_train_loss, all_val_loss)

In [ ]:
'''model.load_state_dict(torch.load("/content/model.pt"))'''

In [ ]:
'''model.eval()
example = torch.rand(1, 21)
traced_script_module = torch.jit.trace(model, example)
traced_script_module.save("/content/andoirdmodel.pt")'''